# 03 - Prompts et mode improved prudent

Objectif : préparer les prompts, harmoniser le warning et tester un mode improved de prudence, non clinique.

Fichiers concernés : `prompts/`, `src/inference.py`, `src/guardrails.py`, `outputs/improved_predictions.csv`.

## Comparaison qualitative des prompts

- Baseline prompt: impose un JSON valide, les trois classes et le warning officiel.
- Improved prompt: ajoute une stratégie de prudence, une préférence pour `uncertain`,
  une vérification de la qualité image et l'interdiction d'affirmations non supportées.
- Uncertainty strict rule: règle de sécurité post-traitement; si la confiance,
  la qualité image ou les observations sont insuffisantes, la sortie devient `uncertain`.

Important: ces prompts documentent une stratégie expérimentale. Tant qu'aucun VLM
réel n'est appelé, le moteur actif reste la baseline jouet déterministe.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "src").exists() and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
SAMPLE_IMAGES_DIR = DATA_DIR / "sample_images"
SRC_DIR = PROJECT_ROOT / "src"
API_DIR = PROJECT_ROOT / "api"
APP_DIR = PROJECT_ROOT / "app"
EVAL_DIR = PROJECT_ROOT / "eval"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"
PROMPTS_DIR = PROJECT_ROOT / "prompts"
TESTS_DIR = PROJECT_ROOT / "tests"
OUTPUTS_DIR.mkdir(exist_ok=True)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

print("PROJECT_ROOT =", PROJECT_ROOT)

PROJECT_ROOT = c:\Users\Sarah Efrei\OneDrive\Desktop\2025-2026\S6\MasterCamp\Code du prof\ARVI-RX


In [ ]:
import pandas as pd
from pathlib import Path
from src.guardrails import WARNING_TEXT, apply_safety_guardrails, validate_prediction
from src.inference import toy_predict

PROMPTS_DIR.mkdir(exist_ok=True)
prompt_texts = {
    "prompt_baseline.txt": f"""You are an educational radiology assistant used in an engineering project.
Return only valid JSON using image_quality, predicted_class, confidence, visual_evidence, justification, limitations and warning.
Classes: normal, suspected_opacity, uncertain.
Do not diagnose, triage, invent history or omit the warning.
Warning: {WARNING_TEXT}
""",
    "prompt_improved.txt": f"""Use a cautious educational strategy.
Check image quality first. Prefer uncertain when evidence is weak, image quality is limited/poor, or confidence would be below 0.60.
Mention only visible visual_evidence and avoid hallucinated medical claims.
Warning: {WARNING_TEXT}
""",
    "prompt_uncertainty_strict.txt": f"""Strict rule:
if confidence < 0.60, predicted_class = uncertain.
if image_quality is limited or poor, predicted_class = uncertain.
if visual_evidence is vague or insufficient, predicted_class = uncertain.
Add a limitation explaining the forced uncertainty.
Warning: {WARNING_TEXT}
""",
}
for name, content in prompt_texts.items():
    path = PROMPTS_DIR / name
    path.write_text(content.strip() + "\n", encoding="utf-8")
    print(path.name, "updated", "warning OK:", WARNING_TEXT in path.read_text(encoding="utf-8"))

def apply_uncertainty_rule(prediction, min_confidence=0.60):
    prediction = dict(prediction)
    conf = float(prediction.get("confidence", 0.0) or 0.0)
    weak_evidence = not prediction.get("visual_evidence")
    limited_quality = prediction.get("image_quality") in {"limited", "poor"}
    if conf < min_confidence or limited_quality or weak_evidence:
        prediction["predicted_class"] = "uncertain"
        prediction["confidence"] = min(conf, min_confidence)
        prediction.setdefault("limitations", []).append(
            "uncertainty forced because confidence, image quality or visual evidence was insufficient"
        )
    return prediction

## Ce que l'improved ajoute réellement

Dans ce notebook, `improved` ne signifie pas meilleur modèle médical. Il signifie:
plus de prudence, seuil d'incertitude, prise en compte de la qualité image et
limitation des affirmations non supportées. Cette stratégie est défendable pour un
prototype responsable, mais elle doit rester séparée de toute promesse clinique.

In [3]:
df = pd.read_csv(DATA_DIR / "synthetic_cases.csv")
rows = []
for _, row in df.iterrows():
    pred = apply_safety_guardrails(apply_uncertainty_rule(toy_predict(PROJECT_ROOT / row["image_path"], mode="improved")))
    valid, errors = validate_prediction(pred)
    rows.append({"case_id": row["case_id"], "filename": Path(row["image_path"]).name, "expected_label": row["label"], "predicted_class": pred["predicted_class"], "confidence": pred["confidence"], "json_valid": valid, "warning_present": bool(pred.get("warning")), "guardrail_errors": ";".join(errors)})
improved_df = pd.DataFrame(rows)
out = OUTPUTS_DIR / "improved_predictions.csv"
improved_df.to_csv(out, index=False)
print(out)
display(improved_df.head())

c:\Users\Sarah Efrei\OneDrive\Desktop\2025-2026\S6\MasterCamp\Code du prof\ARVI-RX\outputs\improved_predictions.csv


,case_id,filename,expected_label,predicted_class,confidence,json_valid,warning_present,guardrail_errors
0,CXR_SYN_001,CXR_SYN_001_normal.png,normal,normal,0.68,True,True,
1,CXR_SYN_002,CXR_SYN_002_suspected_opacity.png,suspected_opacity,suspected_opacity,0.72,True,True,
2,CXR_SYN_003,CXR_SYN_003_uncertain.png,uncertain,uncertain,0.52,True,True,
3,CXR_SYN_004,CXR_SYN_004_normal.png,normal,normal,0.68,True,True,
4,CXR_SYN_005,CXR_SYN_005_suspected_opacity.png,suspected_opacity,suspected_opacity,0.72,True,True,


Conclusion : les prompts sont maintenant plus défendables pour expliquer la
stratégie baseline vs improved. Leur effet réel restera limité tant que le moteur
d'inférence est `toy_predict`; ils préparent surtout la phase pipeline/VLM future.